# Case Study 3: E-Commerce Customer Segmentation

This notebook groups customers using **RFM (Recency, Frequency, Monetary)** features. It applies **K-Means** and **DBSCAN**, reduces the data with **PCA**, visualizes customer groups with **t-SNE**, and suggests marketing actions for each customer persona.

In [ ]:
%pip install -q kagglehub pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import os
import glob
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

In [ ]:
path = kagglehub.dataset_download("carrie1/ecommerce-data")

print("Dataset path:", path)

In [ ]:
csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)

print("CSV files found:", csv_files)

In [ ]:
file_path = csv_files[0]

df = pd.read_csv(file_path, encoding="ISO-8859-1")

df.head()

In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

In [ ]:
df.columns = df.columns.str.strip()

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

df = df.dropna(subset=["CustomerID", "InvoiceDate", "Quantity", "UnitPrice"])
df = df[~df["InvoiceNo"].str.upper().str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df.drop_duplicates()
df["CustomerID"] = df["CustomerID"].astype(int)
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

df.head()

In [ ]:
print("Shape after cleaning:", df.shape)
print("Number of customers:", df["CustomerID"].nunique())
print("Missing values after cleaning:", df.isnull().sum().sum())
print("First invoice date:", df["InvoiceDate"].min())
print("Last invoice date:", df["InvoiceDate"].max())

In [ ]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda dates: (reference_date - dates.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum")
)

rfm.head()

In [ ]:
print("RFM table shape:", rfm.shape)
rfm.describe().round(2)

In [ ]:
rfm[["Recency", "Frequency", "Monetary"]].hist(figsize=(12, 4), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
rfm_features = rfm[["Recency", "Frequency", "Monetary"]].copy()

rfm_log = np.log1p(rfm_features)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

print("Scaled data shape:", rfm_scaled.shape)

In [ ]:
k_values = range(2, 9)
inertia_values = []
silhouette_values = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(rfm_scaled)
    inertia_values.append(model.inertia_)
    silhouette_values.append(
        silhouette_score(
            rfm_scaled,
            labels,
            sample_size=min(2000, len(rfm_scaled)),
            random_state=42
        )
    )

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(list(k_values), inertia_values, marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(list(k_values), silhouette_values, marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.title("Silhouette Score for K-Means")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

rfm["KMeans_Cluster"] = kmeans.fit_predict(rfm_scaled)

rfm.head()

In [ ]:
print("Customers in each K-Means cluster:")
print(rfm["KMeans_Cluster"].value_counts().sort_index())

In [ ]:
kmeans_summary = rfm.groupby("KMeans_Cluster").agg(
    Customer_Count=("KMeans_Cluster", "size"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean")
).round(2)

kmeans_summary

In [ ]:
pca = PCA(n_components=2)
pca_result = pca.fit_transform(rfm_scaled)

pca_data = pd.DataFrame(pca_result, columns=["PCA1", "PCA2"], index=rfm.index)
pca_data["KMeans_Cluster"] = rfm["KMeans_Cluster"]

print("Variance explained by the two PCA components:", round(pca.explained_variance_ratio_.sum(), 3))

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=pca_data,
    x="PCA1",
    y="PCA2",
    hue="KMeans_Cluster",
    palette="tab10",
    s=25,
    alpha=0.7
)
plt.title("K-Means Customer Clusters using PCA")
plt.show()

In [ ]:
dbscan = DBSCAN(eps=0.7, min_samples=10)

dbscan_labels = dbscan.fit_predict(rfm_scaled)
rfm["DBSCAN_Cluster"] = dbscan_labels

print("DBSCAN labels and customer counts:")
print(rfm["DBSCAN_Cluster"].value_counts().sort_index())

number_of_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
print("Number of DBSCAN clusters, excluding noise:", number_of_dbscan_clusters)
print("Number of noise customers:", int((dbscan_labels == -1).sum()))

In [ ]:
pca_data["DBSCAN_Cluster"] = rfm["DBSCAN_Cluster"]

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=pca_data,
    x="PCA1",
    y="PCA2",
    hue="DBSCAN_Cluster",
    palette="tab10",
    s=25,
    alpha=0.7
)
plt.title("DBSCAN Customer Groups using PCA")
plt.show()

In [ ]:
sample_size = min(1500, len(rfm_scaled))
sample_positions = np.random.RandomState(42).choice(
    len(rfm_scaled),
    size=sample_size,
    replace=False
)

sample_scaled = rfm_scaled[sample_positions]
perplexity_value = min(30, max(2, (sample_size - 1) // 3))

tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=perplexity_value,
    init="pca",
    learning_rate="auto"
)

tsne_result = tsne.fit_transform(sample_scaled)

tsne_data = pd.DataFrame(tsne_result, columns=["TSNE1", "TSNE2"])
tsne_data["KMeans_Cluster"] = rfm["KMeans_Cluster"].iloc[sample_positions].to_numpy()
tsne_data["DBSCAN_Cluster"] = rfm["DBSCAN_Cluster"].iloc[sample_positions].to_numpy()

tsne_data.head()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=tsne_data,
    x="TSNE1",
    y="TSNE2",
    hue="KMeans_Cluster",
    palette="tab10",
    s=30,
    alpha=0.75
)
plt.title("K-Means Customer Segments using t-SNE")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=tsne_data,
    x="TSNE1",
    y="TSNE2",
    hue="DBSCAN_Cluster",
    palette="tab10",
    s=30,
    alpha=0.75
)
plt.title("DBSCAN Customer Groups using t-SNE")
plt.show()

In [ ]:
median_recency = rfm["Recency"].median()
median_frequency = rfm["Frequency"].median()
median_monetary = rfm["Monetary"].median()

def get_persona(row):
    if row["Avg_Recency"] <= median_recency and row["Avg_Frequency"] >= median_frequency and row["Avg_Monetary"] >= median_monetary:
        return "Champions"
    elif row["Avg_Recency"] > median_recency and row["Avg_Frequency"] >= median_frequency:
        return "At-Risk Loyal Customers"
    elif row["Avg_Monetary"] >= median_monetary and row["Avg_Frequency"] < median_frequency:
        return "Big-Spender Occasional Buyers"
    elif row["Avg_Recency"] <= median_recency and row["Avg_Frequency"] < median_frequency:
        return "Recent / Low-Frequency Customers"
    else:
        return "Needs Nurturing / Dormant Customers"

In [ ]:
marketing_actions = {
    "Champions": "Reward loyalty, offer early access, and encourage referrals.",
    "At-Risk Loyal Customers": "Send personalized win-back messages and limited-time return offers.",
    "Big-Spender Occasional Buyers": "Recommend related products and offer bundles suited to past purchases.",
    "Recent / Low-Frequency Customers": "Send a welcome sequence and an incentive for a second purchase.",
    "Needs Nurturing / Dormant Customers": "Try a low-cost reactivation campaign and avoid repeated costly offers.",
    "Noise / Unusual Patterns": "Review these customers individually before choosing a campaign."
}

persona_summary = kmeans_summary.copy()
persona_summary["Persona"] = persona_summary.apply(get_persona, axis=1)
persona_summary["Marketing_Action"] = persona_summary["Persona"].map(marketing_actions)

persona_summary

In [ ]:
dbscan_summary = rfm.groupby("DBSCAN_Cluster").agg(
    Customer_Count=("DBSCAN_Cluster", "size"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean")
).round(2)

def get_dbscan_persona(row):
    if row.name == -1:
        return "Noise / Unusual Patterns"
    return get_persona(row)

dbscan_summary["Persona"] = dbscan_summary.apply(get_dbscan_persona, axis=1)
dbscan_summary["Marketing_Action"] = dbscan_summary["Persona"].map(marketing_actions)

dbscan_summary

In [ ]:
comparison = pd.DataFrame({
    "Method": ["K-Means", "DBSCAN"],
    "Number_of_Clusters": [
        rfm["KMeans_Cluster"].nunique(),
        number_of_dbscan_clusters
    ],
    "Noise_Customers": [
        0,
        int((rfm["DBSCAN_Cluster"] == -1).sum())
    ]
})

comparison